# ==============================================
# ASSIGNMENT 1: DECONSTRUCTING THE TRANSFORMER
# ==============================================

# Instructions:
## 1. Use Google Colab for all experiments (free GPU tier is sufficient).
## 2. This notebook provides a complementory solution for all parts of the assignment.

# ==============================================
# SUBMISSION INSTRUCTION
# ==============================================

## 1. Please write the name of the file as `Group_(number)_assignemnt_1_solution.ipynb`

##2. Only one member from one group needs to submit the solution, to avoid any duplicasy.

# PART:1

## Tiny Transformer Implementation

You have to complete the code where it's not completed!


"""
Complete the Code
"""

## Imports and Dataset

**We are using language translation dataset for this task (WMT14 DE-EN dataset)**

- Use first 30k samples for training `[You may Increase the training Samples for better results]`

- Use first 5k samples for validation `[You may Increase the validation Samples for better generalization results]`

- Use first 1k samples for testing.

- We have provided the Helping Functions throughout the notebook, you have to complete the Code and Run as per Questions asked in Assignemnt.

- We have Alreday created the `Dataloaders` to get you started quickely and to make sure resulst can be reproduced. Please do not change the Setup and Import Section.

In [ ]:
# ============================================
# SETUP AND IMPORTS
# ============================================

# Install required packages
!pip install -q torch matplotlib seaborn numpy
!pip install -q transformers datasets tokenizers
!pip install -q sacrebleu

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

import math
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Optional, List, Dict

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================
# DATA LOADING AND PREPROCESSING
# ============================================

# Special tokens
PAD_TOKEN = '<pad>'
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'

# ==============================================================================
#! Change the dataloading samples as asked in the assignemnt.
# ==============================================================================

# ==============================================================================
#! Code Here
# ==============================================================================
# Load WMT14 DE-EN dataset (using a smaller subset)
print("Loading dataset...")
dataset = load_dataset("wmt14", "de-en", split={
    'train': 'train[:10000]',      # Use first ---- samples for training
    'validation': 'validation[:500]',  # Use first ---- for validation
    'test': 'test[:500]'          # Use first ---- for testing
})

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

# Example data point
print("\nExample data point:")
print(dataset['train'][0])

# Build tokenizers using Hugging Face tokenizers library
def build_tokenizer(texts: List[str], vocab_size: int = 10000) -> Tokenizer:
    """Build a simple word-level tokenizer"""
    tokenizer = Tokenizer(WordLevel(unk_token=UNK_TOKEN))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordLevelTrainer(
        vocab_size=vocab_size,
        special_tokens=[PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN],
        min_frequency=2
    )

    tokenizer.train_from_iterator(texts, trainer)
    return tokenizer

# Extract texts for tokenizer training
print("\nBuilding tokenizers...")
de_texts = [item['translation']['de'] for item in dataset['train']]
en_texts = [item['translation']['en'] for item in dataset['train']]

# Build tokenizers
de_tokenizer = build_tokenizer(de_texts, vocab_size=10000)
en_tokenizer = build_tokenizer(en_texts, vocab_size=10000)

print(f"German vocabulary size: {de_tokenizer.get_vocab_size()}")
print(f"English vocabulary size: {en_tokenizer.get_vocab_size()}")

# Get special token IDs
PAD_IDX = en_tokenizer.token_to_id(PAD_TOKEN)
SOS_IDX = en_tokenizer.token_to_id(SOS_TOKEN)
EOS_IDX = en_tokenizer.token_to_id(EOS_TOKEN)
UNK_IDX = en_tokenizer.token_to_id(UNK_TOKEN)

print(f"\nSpecial token IDs:")
print(f"PAD: {PAD_IDX}, SOS: {SOS_IDX}, EOS: {EOS_IDX}, UNK: {UNK_IDX}")

# ============================================
# DATASET CLASS AND DATA PROCESSING
# ============================================

class TranslationDataset(Dataset):
    """Custom dataset for translation"""

    def __init__(self, data, src_tokenizer, tgt_tokenizer, max_length=100):
        self.data = data
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Get source and target sentences
        src_text = self.data[idx]['translation']['de']
        tgt_text = self.data[idx]['translation']['en']

        # Tokenize
        src_tokens = self.src_tokenizer.encode(src_text)
        tgt_tokens = self.tgt_tokenizer.encode(tgt_text)

        # Truncate if necessary
        src_tokens = src_tokens.ids[:self.max_length-2]  # Leave room for SOS/EOS
        tgt_tokens = tgt_tokens.ids[:self.max_length-2]

        # Add SOS and EOS tokens
        src_tokens = [SOS_IDX] + src_tokens + [EOS_IDX]
        tgt_tokens = [SOS_IDX] + tgt_tokens + [EOS_IDX]

        return torch.tensor(src_tokens), torch.tensor(tgt_tokens)

def collate_fn(batch):
    """Custom collate function to pad sequences"""
    src_batch, tgt_batch = [], []

    for src, tgt in batch:
        src_batch.append(src)
        tgt_batch.append(tgt)

    # Pad sequences
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)

    return src_batch, tgt_batch

# Create datasets
print("\nCreating datasets...")
train_dataset = TranslationDataset(dataset['train'], de_tokenizer, en_tokenizer)
val_dataset = TranslationDataset(dataset['validation'], de_tokenizer, en_tokenizer)
test_dataset = TranslationDataset(dataset['test'], de_tokenizer, en_tokenizer)

# Create DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

# Test data loading
print("\nTesting data loading...")
src_batch, tgt_batch = next(iter(train_loader))
print(f"Source batch shape: {src_batch.shape}")
print(f"Target batch shape: {tgt_batch.shape}")

# Show example
print("\nExample tokenized pair:")
src_example = src_batch[0]
tgt_example = tgt_batch[0]

# Decode tokens back to text
src_tokens = [de_tokenizer.id_to_token(idx.item()) for idx in src_example if idx != PAD_IDX]
tgt_tokens = [en_tokenizer.id_to_token(idx.item()) for idx in tgt_example if idx != PAD_IDX]

print(f"Source tokens: {' '.join(src_tokens[:10])}...")
print(f"Target tokens: {' '.join(tgt_tokens[:10])}...")

# Vocabulary mappings for visualization
de_vocab = de_tokenizer.get_vocab()
en_vocab = en_tokenizer.get_vocab()
de_idx2word = {v: k for k, v in de_vocab.items()}
en_idx2word = {v: k for k, v in en_vocab.items()}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.7 MB/s eta 0:00:00
Using device: cuda
Loading dataset...


README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

Train samples: 10000
Validation samples: 500
Test samples: 500

Example data point:
{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}

Building tokenizers...
German vocabulary size: 10000
English vocabulary size: 7216

Special token IDs:
PAD: 0, SOS: 1, EOS: 2, UNK: 3

Creating datasets...

Testing data loading...
Source batch shape: torch.Size([64, 89])
Target batch shape: torch.Size([64, 88])

Example tokenized pair:
Source tokens: <sos> Aber ich kann nicht die bei zahlreichen Gelegenheiten sowohl...
Target tokens: <sos> Nevertheless , I cannot forget the numerous criticisms ,...


# PART:1 (A)

- Implement the following.

  - sinusoidal positional encoding

  -  Multi-head attention

  - scaled-dot product attention

  - Feed-forward layer

  - Encoder and Decoder Layer

  - Encoder and Decoder Block

## Solution 1(A)

In [ ]:
from tqdm import trange


class PositionalEncoding(nn.Module):
    """
    Add positional encoding to embeddings
    PE(pos, 2i) = sin(pos/10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos/10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()

        # TODO: Implement positional encoding


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
        """
        # TODO: Add positional encoding to input


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # TODO: Initialize linear layers for Q, K, V projections and output


        self.dropout = nn.Dropout(dropout)

        # Store attention weights for visualization
        self.attention_weights = None

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            query: (batch_size, seq_len_q, d_model)
            key: (batch_size, seq_len_k, d_model)
            value: (batch_size, seq_len_v, d_model)
            mask: (batch_size, seq_len_q, seq_len_k) or None

        Returns:
            output: (batch_size, seq_len_q, d_model)
        """
        batch_size = query.size(0)

        # TODO: Implement multi-head attention


        # Attention


        # Store weights for visualization


        #  Concatenate heads


        # Final linear projection


        return output

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Calculate scaled dot-product attention
        """
        # TODO: Implement scaled dot-product attention
        # Attention(Q,K,V) = softmax(QK^T/√d_k)V



        return output, attention_weights

class FeedForward(nn.Module):
    """
    Feed-forward network (FFN)
    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()

        # TODO: Implement feed-forward network with 2 linear layrers and dropout


    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # TODO: Implement forward pass
        return

class EncoderLayer(nn.Module):
    """
    Single encoder layer consisting of:
    1. Multi-head self-attention
    2. Feed-forward network
    Both with residual connections and layer normalization
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()

        # TODO: Initialize components


    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Implement encoder layer forward pass
        # Remember: residual connections and layer normalization

        # Self-attention block


        # Feed-forward block


        return x

class DecoderLayer(nn.Module):
    """
    Single decoder layer consisting of:
    1. Masked multi-head self-attention
    2. Multi-head cross-attention
    3. Feed-forward network
    All with residual connections and layer normalization
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()

        # TODO: Initialize components


    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Implement decoder layer forward pass

        # Masked self-attention block


        # Cross-attention block


        # Feed-forward block


        return x

class Encoder(nn.Module):
    """
    Transformer Encoder consisting of multiple encoder layers
    """
    def __init__(self, n_layers: int, d_model: int, n_heads: int,
                 d_ff: int, dropout: float = 0.1):
        super().__init__()

        # TODO: Create stack of encoder layers with normalization


    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Pass input through all encoder layers


        return #TODO

class Decoder(nn.Module):
    """
    Transformer Decoder consisting of multiple decoder layers
    """
    def __init__(self, n_layers: int, d_model: int, n_heads: int,
                 d_ff: int, dropout: float = 0.1):
        super().__init__()

        # TODO: Create stack of decoder layers with niormlaization


    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:

        # TODO: Pass input through all decoder layers


        return #TODO



## PART - 1(B)

Implement a Tiny Transformer with the following specifications:

    - Embedding dimension: 128

    - Transformer Layers 2

    - Configurable number of attention heads (1,2,4..8 etc)

    - Feed-Forward dim: 512

    - Max-Token Seq Length: 128

## Solution 1(B)

In [ ]:
class TinyTransformer(nn.Module):
    """
    Complete Transformer model for translation

    Architecture:
    - Source embedding + positional encoding
    - Target embedding + positional encoding
    - Encoder stack
    - Decoder stack
    - Output projection layer
    """
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int,
                 d_model: int = 256, n_heads: int = 4, n_layers: int = 2,
                 d_ff: int = 1024, max_seq_len: int = 100, dropout: float = 0.1):
        super().__init__()

        # Model parameters
        self.d_model = d_model
        self.n_heads = n_heads

        # TODO: Initialize all components
        # Embeddings

        # Positional encoding


        # Encoder and Decoder stacks


        # Output projection


        # Dropout


        # Initialize parameters
        self._init_parameters()

    def _init_parameters(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def create_src_mask(self, src: torch.Tensor) -> torch.Tensor:
        """Create source mask to hide padding tokens"""

        # TODO: Create mask where True indicates valid position

        return #TODO

    def create_tgt_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        """Create target mask to hide padding tokens and future positions"""
        # TODO: Create mask combining padding mask and causal mask


        # Padding mask


        # Causal mask (lower triangular)


        # Combine masks

        return #TODO

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        """Encode source sequence"""
        # TODO: Implement encoding
        # 1. Embed source tokens
        # 2. Scale embeddings by sqrt(d_model)
        # 3. Add positional encoding
        # 4. Apply dropout
        # 5. Pass through encoder



        return #TODO

    def decode(self, tgt: torch.Tensor, encoder_output: torch.Tensor,
               src_mask: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        """Decode target sequence"""
        # TODO: Implement decoding
        # Similar to encoding but with decoder


        return #TODO

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for training

        Args:
            src: Source sequences (batch_size, src_seq_len)
            tgt: Target sequences (batch_size, tgt_seq_len)

        Returns:
            output: Predicted token logits (batch_size, tgt_seq_len, tgt_vocab_size)
        """
        # TODO: Implement complete forward pass
        # 1. Create masks
        # 2. Encode source
        # 3. Decode target
        # 4. Project to vocabulary


        return #TODO

    def get_attention_weights(self):
        """
        Extract attention weights from all layers for visualization

        Returns dict with:
        - encoder_self_attention: List of attention weights from encoder layers
        - decoder_self_attention: List of attention weights from decoder self-attention
        - decoder_cross_attention: List of attention weights from decoder cross-attention
        """
        attention_weights = {
            'encoder_self_attention': [],
            'decoder_self_attention': [],
            'decoder_cross_attention': []
        }

        # TODO: Extract attention weights from all layers
        # Encoder self-attention


        # Decoder self-attention and cross-attention


        return #TODO

## PART 1(C-D)

- Train multi head model (4 heads) and single head (1) model, by keeping the number of parameters same, adjust attention head dimension accordingly.

- Implement visualization for different attention types like Encoder self-attention, Decoder self-attention and Decoder cross-attention. Visualize the attentions for multi-head and single-head both for given test sentences.

## Solution 1(C-D)

In [ ]:
from tqdm import tqdm, trange

# ============================================
# TODO: WRITE TRAINING FUNCTIONS
# ============================================

def train_epoch(model: nn.Module, dataloader: DataLoader, optimizer: optim.Optimizer,
                criterion: nn.Module, clip: float = 1.0) -> float:
    """Train for one epoch"""
    model.train()

    # TODO

    return # epoch_loss / len(dataloader)

def evaluate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module) -> float:
    """Evaluate model"""
    model.eval()
    epoch_loss = 0

    # TODO

    return # epoch_loss / len(dataloader)

# ============================================
# TODO: WRITE TRANSLATING SENTENCE FUNCTION
# ============================================

def translate_sentence(model: nn.Module, src_sentence: str, max_length: int = 50):
    """Translate a single sentence"""
    model.eval()



    return # translation, tgt_indices

# ============================================
# TODO: WRITE ATTENTION VISUALIZATION FUNCTION
# ============================================


# ========================================================================
# TODO: YOU CAN USE THE ATTENTION VISUALIZATION TOOLS : ADD THE PLOTS HERE
# ========================================================================

def visualize_attention(model: nn.Module, src_sentence: str, tgt_sentence: str = None):
    """
    Visualize attention weights for all heads in all layers
    """
    model.eval()



    # TODO: Implement visualization for different attention types
    # 1. Encoder self-attention
    # 2. Decoder self-attention
    # 3. Decoder cross-attention



# ===================================================
# TODO: WRITE ATTENTION COMPARISON PATTERNS FUNCTION
# ===================================================

def compare_attention_patterns(model_multi: nn.Module, model_single: nn.Module,
                             src_sentence: str):
    """
    Compare attention patterns between multi-head and single-head models
    """
    print("\n=== COMPARING MULTI-HEAD vs SINGLE-HEAD ATTENTION ===")

  # TODO





# ============================================
# TODO: WRITE MAIN TRAINING SCRIPT
# ============================================

def count_parameters(model):
    """Count trainable parameters"""
    return # TODO

# Model configurations
MODEL_CONFIGS = {
    'multi_head': {
        'd_model': 128,
        'n_heads': 4,
        'n_layers': 2,
        'd_ff': 256,
        'dropout': 0.1
    },
    'single_head': {
        'd_model': 128,
        'n_heads': 1,
        'n_layers': 2,
        'd_ff': 456,
        'dropout': 0.1
    }
}

# Initialize models
print("Initializing models...")

model_multi = TinyTransformer(
    src_vocab_size=len(de_vocab),
    tgt_vocab_size=len(en_vocab),
    **MODEL_CONFIGS['multi_head']
).to(device)

model_single = TinyTransformer(
    src_vocab_size=len(de_vocab),
    tgt_vocab_size=len(en_vocab),
    **MODEL_CONFIGS['single_head']
).to(device)

print(f"Multi-head model parameters: {count_parameters(model_multi):,}")
print(f"Single-head model parameters: {count_parameters(model_single):,}")

# Training settings
LEARNING_RATE = 0.0001 # YOU CAN CHOOSE AS PER TRAINING AND VALIDATION PLOTS
N_EPOCHS = # TODO

# Loss function - ignore padding token
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ============================================
# TRAINING LOOP FOR MULTI-HEAD MODEL
# ============================================

print("\n" + "="*50)
print("TRAINING MULTI-HEAD MODEL (4 heads)")
print("="*50)

optimizer_multi = optim.Adam(model_multi.parameters(), lr=LEARNING_RATE)

# Training history
train_losses_multi = []
val_losses_multi = []

best_val_loss = float('inf')

for epoch in trange(N_EPOCHS, desc="Epochs"):
    start_time = time.time()

    # Train
    train_loss = train_epoch(model_multi, train_loader, optimizer_multi, criterion)

    # Evaluate
    val_loss = evaluate(model_multi, val_loader, criterion)

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    train_losses_multi.append(train_loss)
    val_losses_multi.append(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_multi.state_dict(), 'tiny_transformer_multi.pt')

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins:.0f}m {epoch_secs:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

# Load best model
model_multi.load_state_dict(torch.load('tiny_transformer_multi.pt'))

# ============================================
# TODO: TRAINING LOOP FOR SINGLE-HEAD MODEL
# ============================================

print("\n" + "="*50)
print("TRAINING SINGLE-HEAD MODEL")
print("="*50)

optimizer_single = optim.Adam(model_single.parameters(), lr=LEARNING_RATE)

# Training history
train_losses_single = []
val_losses_single = []

best_val_loss = # TODO

for epoch in trange(N_EPOCHS):
    start_time = time.time()

    # Train


    # Evaluate




    # Save best model


    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins:.0f}m {epoch_secs:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

# Load best model
# TODO


# ============================================
# TODO: PLOT TRAINING CURVES
# ============================================

plt.figure(figsize=(12, 5))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(train_losses_multi, label='Multi-head Train')
plt.plot(val_losses_multi, label='Multi-head Val')
plt.plot(train_losses_single, label='Single-head Train')
plt.plot(val_losses_single, label='Single-head Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# Validation loss comparison
plt.subplot(1, 2, 2)
epochs = range(1, N_EPOCHS + 1)
plt.plot(epochs, val_losses_multi, 'o-', label='Multi-head')
plt.plot(epochs, val_losses_single, 's-', label='Single-head')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Loss Comparison')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ============================================
# TODO: EVALUATION AND VISUALIZATION
# ============================================

print("\n" + "="*50)
print("EVALUATION ON TEST SENTENCES")
print("="*50)

# Test sentences
test_sentences = [
    "Ein Mann läuft auf der Straße.",  # A man walks on the street
    "Die Katze sitzt auf dem Tisch.",   # The cat sits on the table
    "Ich liebe dich.",                  # I love you
    "Das Wetter ist heute schön.",      # The weather is nice today
    "Können Sie mir helfen?"            # Can you help me?
]

# Translate with both models
print("\nTranslation Examples:")
for i, src in enumerate(test_sentences):
    print(f"\n{i+1}. Source: {src}")

    trans_multi, _ = translate_sentence(model_multi, src)
    print(f"   Multi-head: {trans_multi}")

    trans_single, _ = translate_sentence(model_single, src)
    print(f"   Single-head: {trans_single}")

# ============================================
# ATTENTION VISUALIZATION FOR TEST SENTENCE
# ============================================

print("\n" + "="*50)
print("ATTENTION VISUALIZATION")
print("="*50)

# Choose a test sentence for detailed visualization
test_sentence = "Die Katze sitzt auf dem Tisch."
print(f"\nVisualizing attention for: '{test_sentence}'")

# Visualize multi-head model attention
print("\n### MULTI-HEAD MODEL ###")
visualize_attention(model_multi, test_sentence)

# Visualize single-head model attention
print("\n### SINGLE-HEAD MODEL ###")
visualize_attention(model_single, test_sentence)

# Compare attention patterns
compare_attention_patterns(model_multi, model_single, test_sentence)



# PART: 2 - Architecture Ablation Study


## PART 2(A)

**Study the Role of Residual Connections**

## Solution 2(A)

In [ ]:
# =========================================================================================================================
# TODO: ARCHITECTURAL ABLATION STUDIES [YOU MAY CHOOSE DIFFERENT IMPLEMENTATION STRATEGY AS LONG AS YOU IMPLEMENT AS ASKED]
# =========================================================================================================================

# --- Experiment 1: Removing Residual Connections ---

class EncoderLayerNoResidual(EncoderLayer):
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:

       # TODO

        return # TODO

class DecoderLayerNoResidual(DecoderLayer):
    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: Optional[torch.Tensor] = None, tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO
        return # TODO

class EncoderNoResidual(Encoder):
    def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__(n_layers, d_model, n_heads, d_ff, dropout)
        # TODO

class DecoderNoResidual(Decoder):
     def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__(n_layers, d_model, n_heads, d_ff, dropout)
        # TODO

class TransformerNoResidual(TinyTransformer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # TODO

model_no_residual = # TODO: INITIALIZE THE MODEL

# TODO -- RUN TRAINING

## IMPLEMNT **Learnable skip weights** AND TRAIN AGAIN TO OBSERVE THE CHANGES.

In [ ]:
# --- Experiment 2: IMPLEMENT LEARNABLE SKIP WEIGHTS ---


# TODO: WRITE THE CODE SIMILIARLY




## IMPLEMNT **Long-Range Skip Connections** AND TRAIN AGAIN TO OBSERVE THE CHANGES.

In [ ]:
# --- Experiment 3: IMPLEMENT LONG RANGE SKIP CONNECTION ---


# TODO: WRITE THE CODE SIMILIARLY




## PART 2(B)

**Study the Role of Feed-Forward Layers**

- Answer the questions as asked in the Assignemnt.

## Solution 2(B)

In [ ]:
# --- Experiment 1: Removing Feed-Forward Layers ---

class EncoderLayerNoFFN(EncoderLayer):
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:

        return  # TODO # Skip FFN

class DecoderLayerNoFFN(DecoderLayer):
    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: Optional[torch.Tensor] = None, tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:

        return # TODO

class EncoderNoFFN(Encoder):
    def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__(n_layers, d_model, n_heads, d_ff, dropout)
        # TODO

class DecoderNoFFN(Decoder):
     def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__(n_layers, d_model, n_heads, d_ff, dropout)
        # TODO

class TransformerNoFFN(TinyTransformer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # TODO

# TODO

# Answer the questions by training the model

# PART:3 (Attention Modulation)

 - Implement Token Distance as an Attention Bias (as asked in assignemnt)

## Soultion 3(A)

In [ ]:
# ==============================================================================
# TOOD: ATTENTION MODULATION (Token Distance Bias)
# ==============================================================================

class DistanceAwareMultiHeadAttention(MultiHeadAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1, max_seq_len: int = 128):
        super().__init__(d_model, n_heads, dropout)
        # Create a distance bias matrix that is not a model parameter
        # TOOD

        # We can use a learnable parameter to scale the distance penalty
        # TOOD) # Negative to penalize distance

        # Create a bias tensor. We use the absolute distance.
        # Think how could you make penalty less harsh for closer tokens.

        # TOOD

    def scaled_dot_product_attention(self, Q, K, V, mask=None):

        # TOOD


        return # output, attention_weights

class EncoderLayerDistanceAware(EncoderLayer):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1, max_seq_len: int = 128):
        super().__init__(d_model, n_heads, d_ff, dropout)
        # TOOD

class EncoderDistanceAware(Encoder):
    def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1, max_seq_len: int = 128):
        super().__init__(n_layers, d_model, n_heads, d_ff, dropout)
        # TOOD

class TransformerDistanceAware(TinyTransformer):
     def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Only modify the encoder for this experiment for simplicity (You may choose)
        # TOOD

## Solution 3(B)

In [ ]:

# TODO

# Train the Model

# --- Plot Final Comparison between validation loss 'Baseline (Multi-head)' and 'Distance-Aware Attention' ---

# TODO


print("\nAssignment complete.")